# 填充和步幅

## 填充

如果在卷积的过程中不填充步幅，图像在卷积之后尺寸会变小（丢失边缘像素），所以我们需要在输入图像的边缘填充0，以此来保持卷积之后的图像尺寸不发生变化

input:$n_h \times n_w$

kernel:$k_h \times k_w$

output:$n_h - k_h + 1 \times n_w - k_w + 1$

padding:$p_h = k_h -1, p_w = k_w -1$(两侧对称填充)

下面建立一个宽度和高度都为3的二维卷积层，并在所有侧边填充1个像素。给定高度和宽度为8的输入，则输出的高度和宽度也是8。

In [ ]:
import torch
from torch import nn


# 为了方便起见，我们定义了一个计算卷积层的函数。
# 此函数初始化卷积层权重，并对输入和输出提高和缩减相应的维数
def comp_conv2d(conv2d, X):
    # 这里的（1，1）表示批量大小和通道数都是1
    X = X.reshape((1, 1) + X.shape)  # 这里是元组的拼接操作
    Y = conv2d(X)
    # 省略前两个维度：批量大小和通道
    return Y.reshape(Y.shape[2:])

# 请注意，这里每边都填充了1行或1列，因此总共添加了2行或2列
conv2d = nn.Conv2d(1, 1, kernel_size=3, padding=1)  # 参数依次表示为输入通道数、输出通道数、卷积核大小、填充步幅
# 需要注意的是padding是填充到输入图像的边缘的，而不是填充到卷积核的边缘,这里的padding是两侧对称填充
# kernel默认是随机初始化，padding如果没有指定，默认是0
X = torch.rand(size=(8, 8))
comp_conv2d(conv2d, X).shape

c:\Users\20249\.conda\envs\test1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch.Size([8, 8])

kernel的高度和宽度不一样的时候，可以填充不同的高度以及宽度，使得输出和输入具有相同的高度和宽度，下面我们使用高度为5，宽度为3的卷积核，高度和宽度两边分别填充2和1

In [2]:
conv2d = nn.Conv2d(1, 1, kernel_size=(5, 3), padding=(2, 1)) # padding可以传入元组，表示两边分别去填充
comp_conv2d(conv2d, X).shape


torch.Size([8, 8])

## Stride(步幅)

设置步幅本质上是缩减采样次数：

output:$(n_h - k_h + p_h + 1) / stride \times (n_w - k_w + p_w + 1) / stride$

默认的设置是：

padding : $p_w = p_h $

stride: $s_w = s_h$


# 多输入多输出通道

## 多通道输入

多输入通道（如彩色的RGB图像），需要构造多个卷积核，每个卷积核对应一个通道，求取完卷积之后相加得到一个输出的矩阵

In [3]:
import torch
from d2l import torch as d2l

def corr2d_multi_in(X, K):
    # 先遍历“X”和“K”的第0个维度（通道维度），再把它们加在一起
    return sum(d2l.corr2d(x, k) for x, k in zip(X, K))  # zip函数将输入通道和卷积核对应起来
X = torch.tensor([[[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]],
               [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]]])
K = torch.tensor([[[0.0, 1.0], [2.0, 3.0]], [[1.0, 2.0], [3.0, 4.0]]])

corr2d_multi_in(X, K)

tensor([[ 56.,  72.],
        [104., 120.]])

## 多通道输出
本质上是对kernel再进行一次升维，每个chanel对应一组的卷积核

In [4]:
def corr2d_multi_out(X, K):
    # 先遍历“X”和“K”的第0个维度（通道维度），再把它们加在一起
    return torch.stack([corr2d_multi_in(X, k) for k in K], 0)  #  stack函数表示在第0个维度上进行堆叠
'''
outputs = [output0, output1, output2]  # 每个都是 (H, W)
result = torch.stack(outputs, 0)       # → 形状变为 (3, H, W)
'''
K = torch.stack((K, K+1, K+2), 0)
K.shape

torch.Size([3, 2, 2, 2])

下面对输入张量与卷积核张量执行互相关运算，现在的输出包含3个通道

In [5]:
corr2d_multi_out(X, K)

tensor([[[ 56.,  72.],
         [104., 120.]],

        [[ 76., 100.],
         [148., 172.]],

        [[ 96., 128.],
         [192., 224.]]])

# $1 \times 1 卷积层$

$1\times 1$卷积层本质上是一种全连接层，他只是去做通道融合，不去识别空间的模式

在数学的模式上， $1\times 1$卷积层和全连接层的数学形式是一模一样的：

假设输入的特征图形状为 $C_{in} \times H \times W$（输入通道数 $\times$ 高 $\times$ 宽）。

如果我们固定住空间位置，只看某一个特定的像素点 $(i, j)$，那么在这个点上，输入数据可以被抽出来看作一个长度为 $C_{in}$ 的一维向量：
$$\mathbf{x} = [X_{1, i, j}, X_{2, i, j}, \dots, X_{C_{in}, i, j}]^T$$


现在我们用 $C_{out}$ 个大小为 $1\times1 \times C_{in}$ 的卷积核去和它做互相关运算：每一个卷积核在空间上只有 1 个权重，但在通道方向上有 $C_{in}$ 个权重。因此，第 $k$ 个卷积核可以表示为一个长度为 $C_{in}$ 的向量 $\mathbf{w}_k$。（W权重为 （$C_{out} \times C_{in}$））

运算结果就是向量内积：$y_k = \mathbf{w}_k \cdot \mathbf{x} + b_k$。

把所有 $C_{out}$ 个卷积核组合起来，权重就变成了一个大小为 $C_{out} \times C_{in}$ 的权重矩阵 $W$，偏置为 $\mathbf{b}$。整个像素点 $(i, j)$ 处的输出向量 $\mathbf{y}$ 就是：$$\mathbf{y} = W\mathbf{x} + \mathbf{b}$$

下面用全连接层去实现$1 \times 1$卷积层,需要对输入和输出的数据形状进行调整

In [ ]:
def corr2d_multi_in_out_1x1(X, K):
    c_i, h, w = X.shape
    c_o = K.shape[0]
    X = X.reshape((c_i, h * w))  # 相当于对输入数据X进行降维
    K = K.reshape((c_o, c_i))
    # 全连接层中的矩阵乘法
    Y = torch.matmul(K, X)
    return Y.reshape((c_o, h, w))